# Arabic ↔ English Machine Translation

**CAI350 – Natural Language Processing academic team project**

This notebook implements bidirectional Arabic–English machine translation using BLOOMZ-560M and Helsinki-NLP Opus-MT, with OPUS100 preprocessing, Gradio demos, and translation-quality evaluation using BLEU, ChrF, BERTScore, and COMET.

> The evaluation scores retained in this notebook and reported in the project report are for **BLOOMZ**. Opus-MT was implemented as a functional translation model and interactive demo, but it was not included in the reported metric comparison.

In [ ]:
# Install required libraries
%pip install datasets sentencepiece transformers peft evaluate accelerate bitsandbytes sacrebleu gradio bert_score

import pandas as pd
import re
import sentencepiece as spm
from datasets import load_dataset
import evaluate
import gradio as gr

## 1. Dataset Preparation

Load the OPUS100 Arabic–English parallel corpus, clean sentence pairs, filter sentence length, and train SentencePiece BPE tokenizers for both language directions.

In [ ]:
# This function loads OPUS100 dataset, cleans it, applies tokenization, and saves preprocessed CSV
def process_dataset(language_pair):
    # Load Arabic-English OPUS100 dataset
    dataset = load_dataset("opus100", "ar-en")
    df = pd.DataFrame(dataset["train"])

    # Set source/target based on language direction
    if language_pair == "en-ar":
        df["source"] = df["translation"].apply(lambda x: x["en"])
        df["target"] = df["translation"].apply(lambda x: x["ar"])
    elif language_pair == "ar-en":
        df["source"] = df["translation"].apply(lambda x: x["ar"])
        df["target"] = df["translation"].apply(lambda x: x["en"])
    else:
        raise ValueError("Invalid language pair")

    df.drop(columns=["translation"], inplace=True)

    # Clean text: remove noise, normalize spacing
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.strip()
        text = re.sub(r"[^\w\s.!?]", "", text)
        return text

    df["source"] = df["source"].apply(clean_text)
    df["target"] = df["target"].apply(clean_text)

    # Remove empty and duplicate pairs
    df.dropna(inplace=True)
    df.drop_duplicates(subset=["source", "target"], inplace=True)

    # Filter by sentence length (3 to 50 words)
    def filter_sentence(sentence):
        words = sentence.split()
        return 3 <= len(words) <= 50

    df = df[df["source"].apply(filter_sentence)]
    df = df[df["target"].apply(filter_sentence)]

    # Normalize text: lowercase + remove punctuation
    df["source"] = df["source"].str.lower().apply(lambda x: re.sub(r"[^\w\s]", "", x))
    df["target"] = df["target"].str.lower().apply(lambda x: re.sub(r"[^\w\s]", "", x))

    # Train SentencePiece (BPE) models
    def train_sentencepiece(texts, prefix):
        with open("train_data.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(texts))
        spm.SentencePieceTrainer.train(
            input="train_data.txt",
            model_prefix=prefix,
            vocab_size=8000,
            model_type="bpe"
        )

    train_sentencepiece(df["source"].tolist(), f"bpe_{language_pair.split('-')[0]}")
    train_sentencepiece(df["target"].tolist(), f"bpe_{language_pair.split('-')[1]}")

    # Load models and tokenize
    sp_source = spm.SentencePieceProcessor()
    sp_source.load(f"bpe_{language_pair.split('-')[0]}.model")
    sp_target = spm.SentencePieceProcessor()
    sp_target.load(f"bpe_{language_pair.split('-')[1]}.model")

    df_clean = df.copy()
    df_clean.to_csv(f"cleaned_opus_plain_{language_pair}.csv", index=False)#before BPE

    df["source"] = df["source"].apply(lambda x: " ".join(sp_source.encode_as_pieces(x)))
    df["target"] = df["target"].apply(lambda x: " ".join(sp_target.encode_as_pieces(x)))

    # Save preprocessed dataset
    df.to_csv(f"cleaned_opus_{language_pair}.csv", index=False)
    print(f"Data for {language_pair} processed and saved!")

# Run preprocessing for both directions
process_dataset("en-ar")
process_dataset("ar-en")


In [3]:
try:
    df_en_ar = pd.read_csv("cleaned_opus_en-ar.csv")
    df_ar_en = pd.read_csv("cleaned_opus_ar-en.csv")

    print("Head of cleaned_opus_en-ar.csv:")
    display(df_en_ar.head())

    print("\nHead of cleaned_opus_ar-en.csv:")
    display(df_ar_en.head())

except FileNotFoundError:
    print("Error: One or both of the CSV files were not found. Please run the dataset processing code first.")


Head of cleaned_opus_en-ar.csv:


,source,target
0,▁what ▁is ▁she ▁doing ▁here,▁ما ▁الذي ▁تفعله ▁هناك
1,▁i ▁dont ▁like ▁it,▁لا ▁أحب ▁ذلك
2,▁did ▁you ▁get ▁the ▁part,▁هل ▁حصلت ▁على ▁جزء
3,▁its ▁none ▁of ▁your ▁business,▁هذا ▁ليس ▁من ▁شأن ك
4,▁i ▁was ▁wrong,▁كنت ▁على ▁خطأ


Head of cleaned_opus_ar-en.csv:


,source,target
0,▁ما ▁الذي ▁تفعله ▁هناك,▁what ▁is ▁she ▁doing ▁here
1,▁لا ▁أحب ▁ذلك,▁i ▁dont ▁like ▁it
2,▁هل ▁حصلت ▁على ▁جزء,▁did ▁you ▁get ▁the ▁part
3,▁هذا ▁ليس ▁من ▁شأن ك,▁its ▁none ▁of ▁your ▁business
4,▁كنت ▁على ▁خطأ,▁i ▁was ▁wrong


## 2. BLOOMZ Zero-Shot Translation

Use `bigscience/bloomz-560m` with natural-language translation prompts for Arabic ↔ English inference.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load BLOOMZ tokenizer and model
bloomz_model_name = "bigscience/bloomz-560m"
bloomz_tokenizer = AutoTokenizer.from_pretrained(bloomz_model_name)
bloomz_model = AutoModelForCausalLM.from_pretrained(bloomz_model_name)

In [11]:
def translate_bloomz(text, direction):
    src_lang, tgt_lang = direction.split(" to ")
    prompt = f"Translate this from {src_lang} to {tgt_lang}: {text}"
    inputs = bloomz_tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    bloomz_model.to(inputs["input_ids"].device)
    outputs = bloomz_model.generate(**inputs, max_new_tokens=100)
    decoded = bloomz_tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned = decoded.replace(prompt, "").strip()
    return cleaned

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Arabic ⇄ English Translator")

    direction = gr.State("Arabic to English")

    with gr.Row():
        input_text = gr.Textbox(label="Arabic", lines=4)
        output_text = gr.Textbox(label="English", lines=4)

    with gr.Row():
        swap_btn = gr.Button("Swap Languages")
        translate_btn = gr.Button("Translate")

    def swap_langs(current_direction):
        if current_direction == "Arabic to English":
            return "English to Arabic", gr.update(label="English"), gr.update(label="Arabic")
        else:
            return "Arabic to English", gr.update(label="Arabic"), gr.update(label="English")

    swap_btn.click(
        fn=swap_langs,
        inputs=direction,
        outputs=[direction, input_text, output_text]
    )

    translate_btn.click(
        fn=translate_bloomz,
        inputs=[input_text, direction],
        outputs=output_text
    )

demo.launch()

## 3. BLOOMZ Evaluation

Evaluate the first 100 cleaned sentence pairs in each direction using BLEU, ChrF, and BERTScore. COMET is added in the later evaluation cell.

In [13]:
bleu = evaluate.load("bleu")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

In [14]:
df_ar_en = pd.read_csv("cleaned_opus_plain_ar-en.csv").head(100)
df_en_ar = pd.read_csv("cleaned_opus_plain_en-ar.csv").head(100)

In [15]:
sources_ar = df_ar_en["source"].tolist()
references_en = [[t] for t in df_ar_en["target"].tolist()]
predictions_en = [translate_bloomz(text, direction="Arabic to English") for text in sources_ar]

bleu_ar_en = bleu.compute(predictions=predictions_en, references=references_en)
chrf_ar_en = chrf.compute(predictions=predictions_en, references=[r[0] for r in references_en])
bertscore_ar_en = bertscore.compute(predictions=predictions_en, references=[r[0] for r in references_en], lang="en")

print("Arabic → English")
print("BLEU:", bleu_ar_en["bleu"])
print("ChrF:", chrf_ar_en["score"])
print("BERTScore (F1):", sum(bertscore_ar_en["f1"]) / len(bertscore_ar_en["f1"]))

for i in range(5):
    print("Arabic:", sources_ar[i])
    print("Reference EN:", references_en[i][0])
    print("Predicted EN:", predictions_en[i])
    print("---")
print()


sources_en = df_en_ar["source"].tolist()
references_ar = [[t] for t in df_en_ar["target"].tolist()]
predictions_ar = [translate_bloomz(text, direction="English to Arabic") for text in sources_en]

bleu_en_ar = bleu.compute(predictions=predictions_ar, references=references_ar)
chrf_en_ar = chrf.compute(predictions=predictions_ar, references=[r[0] for r in references_ar])
bertscore_en_ar = bertscore.compute(predictions=predictions_ar, references=[r[0] for r in references_ar], lang="ar")

print("English → Arabic")
print("BLEU:", bleu_en_ar["bleu"])
print("ChrF:", chrf_en_ar["score"])
print("BERTScore (F1):", sum(bertscore_en_ar["f1"]) / len(bertscore_en_ar["f1"]))
for i in range(5):
    print("English:", sources_en[i])
    print("Reference AR:", references_ar[i][0])
    print("Predicted AR:", predictions_ar[i])
    print("---")

Arabic → English
BLEU: 0.07899311742910724
ChrF: 28.261717434852745
BERTScore (F1): 0.8385115671157837
Arabic: ما الذي تفعله هناك
Reference EN: what is she doing here
Predicted EN: ? What are you doing there?
---
Arabic: لا أحب ذلك
Reference EN: i dont like it
Predicted EN: . I don't like that.
---
Arabic: هل حصلت على جزء 
Reference EN: did you get the part
Predicted EN: of the money  I got It was a part of the money I got.
---
Arabic: هذا ليس من شأنك
Reference EN:  its none of your business
Predicted EN: . This isn't your business.
---
Arabic: كنت على خطأ
Reference EN: i was wrong
Predicted EN: . I was wrong.
---

English → Arabic
BLEU: 0.03668242794944664
ChrF: 18.171398735989577
BERTScore (F1): 0.48483649849891663
English: what is she doing here
Reference AR: ما الذي تفعله هناك
Predicted AR: ؟ ماذا تفعل هنا؟
---
English: i dont like it
Reference AR: لا أحب ذلك
Predicted AR: 
---
English: did you get the part
Reference AR: هل حصلت على جزء 
Predicted AR: ؟ هل حصلت على الجزء؟
---
Engli

## 4. Opus-MT Translation

Use Helsinki-NLP MarianMT checkpoints specialized for English→Arabic and Arabic→English translation.

In [ ]:
#Helsinki-NLP/opus-mt model
from transformers import MarianMTModel, MarianTokenizer
import gradio as gr

model_names = {
    "English to Arabic": "Helsinki-NLP/opus-mt-en-ar",
    "Arabic to English": "Helsinki-NLP/opus-mt-ar-en"
}


models = {}
tokenizers = {}
for direction, name in model_names.items():
    tokenizers[direction] = MarianTokenizer.from_pretrained(name)
    models[direction] = MarianMTModel.from_pretrained(name)


def translate_opus(text, direction):
    tokenizer = tokenizers[direction]
    model = models[direction]
    tokens = tokenizer(text, return_tensors="pt", padding=True)
    output = model.generate(**tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Arabic ⇄ English Translator (Opus-MT Style)")

    direction = gr.State("Arabic to English")

    with gr.Row():
        input_text = gr.Textbox(label="Arabic", lines=4)
        output_text = gr.Textbox(label="English", lines=4)

    with gr.Row():
        swap_btn = gr.Button("Swap Languages")
        translate_btn = gr.Button("Translate")

    def swap_dir(current_dir):
        if current_dir == "Arabic to English":
            return "English to Arabic", gr.update(label="English"), gr.update(label="Arabic")
        else:
            return "Arabic to English", gr.update(label="Arabic"), gr.update(label="English")

    swap_btn.click(fn=swap_dir, inputs=direction, outputs=[direction, input_text, output_text])
    translate_btn.click(fn=translate_opus, inputs=[input_text, direction], outputs=output_text)

demo.launch()

In [ ]:
model_name_en_ar = "Helsinki-NLP/opus-mt-en-ar"
model_name_ar_en = "Helsinki-NLP/opus-mt-ar-en"

tokenizer_en_ar = MarianTokenizer.from_pretrained(model_name_en_ar)
model_en_ar = MarianMTModel.from_pretrained(model_name_en_ar)

tokenizer_ar_en = MarianTokenizer.from_pretrained(model_name_ar_en)
model_ar_en = MarianMTModel.from_pretrained(model_name_ar_en)

def translate_opus(text, direction):
    if direction == "Arabic to English":
        tokenizer = tokenizer_ar_en
        model = model_ar_en
    else:
        tokenizer = tokenizer_en_ar
        model = model_en_ar

    tokens = tokenizer(text, return_tensors="pt", padding=True)
    output = model.generate(**tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)



## 5. COMET Evaluation and Final Metric Summary

Load COMET and compute the final BLOOMZ evaluation summary for both translation directions.

In [ ]:
!pip install unbabel-comet


In [21]:
import os
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from comet import download_model, load_from_checkpoint

# تحميل نموذج COMET
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# Arabic → English
sources_ar = df_ar_en["source"].tolist()
references_en = [[t] for t in df_ar_en["target"].tolist()]
predictions_en = [translate_bloomz(text, direction="Arabic to English") for text in sources_ar]

bleu_ar_en = bleu.compute(predictions=predictions_en, references=references_en)
chrf_ar_en = chrf.compute(predictions=predictions_en, references=[r[0] for r in references_en])
bertscore_ar_en = bertscore.compute(predictions=predictions_en, references=[r[0] for r in references_en], lang="en")
comet_data_ar_en = [{"src": src, "mt": pred, "ref": ref[0]} for src, pred, ref in zip(sources_ar, predictions_en, references_en)]
comet_result_ar_en = comet_model.predict(comet_data_ar_en, batch_size=8, gpus=1)
comet_ar_en_score = sum(comet_result_ar_en.scores) / len(comet_result_ar_en.scores)

print("Arabic → English")
for i in range(5):
    print("Arabic:", sources_ar[i])
    print("Reference EN:", references_en[i][0])
    print("Predicted EN:", predictions_en[i])
    print("---")
print("BLEU:", bleu_ar_en["bleu"])
print("ChrF:", chrf_ar_en["score"])
print("BERTScore (F1):", sum(bertscore_ar_en["f1"]) / len(bertscore_ar_en["f1"]))
print("COMET:", comet_ar_en_score)

# English → Arabic
sources_en = df_en_ar["source"].tolist()
references_ar = [[t] for t in df_en_ar["target"].tolist()]
predictions_ar = [translate_bloomz(text, direction="English to Arabic") for text in sources_en]

bleu_en_ar = bleu.compute(predictions=predictions_ar, references=references_ar)
chrf_en_ar = chrf.compute(predictions=predictions_ar, references=[r[0] for r in references_ar])
bertscore_en_ar = bertscore.compute(predictions=predictions_ar, references=[r[0] for r in references_ar], lang="ar")
comet_data_en_ar = [{"src": src, "mt": pred, "ref": ref[0]} for src, pred, ref in zip(sources_en, predictions_ar, references_ar)]
comet_result_en_ar = comet_model.predict(comet_data_en_ar, batch_size=8, gpus=1)
comet_en_ar_score = sum(comet_result_en_ar.scores) / len(comet_result_en_ar.scores)

print("\nEnglish → Arabic")
for i in range(5):
    print("English:", sources_en[i])
    print("Reference AR:", references_ar[i][0])
    print("Predicted AR:", predictions_ar[i])
    print("---")
print("BLEU:", bleu_en_ar["bleu"])
print("ChrF:", chrf_en_ar["score"])
print("BERTScore (F1):", sum(bertscore_en_ar["f1"]) / len(bertscore_en_ar["f1"]))
print("COMET:", comet_en_ar_score)

# Summary
print("\n--- Summary ---")
print(f"BLEU (AR→EN): {bleu_ar_en['bleu']:.4f} | BLEU (EN→AR): {bleu_en_ar['bleu']:.4f}")
print(f"ChrF (AR→EN): {chrf_ar_en['score']:.4f} | ChrF (EN→AR): {chrf_en_ar['score']:.4f}")
print(f"BERTScore F1 (AR→EN): {sum(bertscore_ar_en['f1']) / len(bertscore_ar_en['f1']):.4f} | "
      f"BERTScore F1 (EN→AR): {sum(bertscore_en_ar['f1']) / len(bertscore_en_ar['f1']):.4f}")
print(f"COMET (AR→EN): {comet_ar_en_score:.4f} | COMET (EN→AR): {comet_en_ar_score:.4f}")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Arabic → English
Arabic: ما الذي تفعله هناك
Reference EN: what is she doing here
Predicted EN: ? What are you doing there?
---
Arabic: لا أحب ذلك
Reference EN: i dont like it
Predicted EN: . I don't like that.
---
Arabic: هل حصلت على جزء 
Reference EN: did you get the part
Predicted EN: of the money  I got It was a part of the money I got.
---
Arabic: هذا ليس من شأنك
Reference EN:  its none of your business
Predicted EN: . This isn't your business.
---
Arabic: كنت على خطأ
Reference EN: i was wrong
Predicted EN: . I was wrong.
---
BLEU: 0.07899311742910724
ChrF: 28.261717434852745
BERTScore (F1): 0.8385115671157837
COMET: 0.6770707699656486


English → Arabic
English: what is she doing here
Reference AR: ما الذي تفعله هناك
Predicted AR: ؟ ماذا تفعل هنا؟
---
English: i dont like it
Reference AR: لا أحب ذلك
Predicted AR: 
---
English: did you get the part
Reference AR: هل حصلت على جزء 
Predicted AR: ؟ هل حصلت على الجزء؟
---
English:  its none of your business
Reference AR: هذا ليس من شأنك
Predicted AR: . ليس من شأنك.
---
English: i was wrong
Reference AR: كنت على خطأ
Predicted AR: في ذلك. كان ذلك خطأً.
---
BLEU: 0.03668242794944664
ChrF: 18.171398735989577
BERTScore (F1): 0.48483649849891663
COMET: 0.5955624157190322

--- Summary ---
BLEU (AR→EN): 0.0790 | BLEU (EN→AR): 0.0367
ChrF (AR→EN): 28.2617 | ChrF (EN→AR): 18.1714
BERTScore F1 (AR→EN): 0.8385 | BERTScore F1 (EN→AR): 0.4848
COMET (AR→EN): 0.6771 | COMET (EN→AR): 0.5956
